# FRE 521D: Data Analytics in Climate, Food and Environment
## Lab 5: Data Cleaning II - Outliers, Validation, and Quality Scoring

**Program:** UBC Master of Food and Resource Economics  
**Instructor:** Asif Ahmed Neloy

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E6A23C; padding: 15px; margin: 15px 0;">
    <h3 style="margin-top: 0; color: #856404;">Submission Deadline</h3>
    <p style="margin-bottom: 0; font-size: 1.2em;"><strong>End of Day: Thursday, February 05, 2026</strong></p>
</div>

---

## Lab Objectives

1. Implement IQR and Z-score outlier detection methods
2. Build domain-based validation rules
3. Create a data quality scoring system

---

## Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 15)
print("Setup complete!")

# Load data
df_weather = pd.read_csv('../../Datasets/GlobalWeatherRepository.csv')
print(f"Weather Data: {df_weather.shape[0]:,} rows, {df_weather.shape[1]} columns")

Setup complete!
Weather Data: 43,884 rows, 41 columns


In [2]:
df_weather

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,...,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.5200,69.1800,Asia/Kabul,1715849100,2024-05-16 13:15,...,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.3300,19.8200,Europe/Tirane,1715849100,2024-05-16 10:45,...,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.7600,3.0500,Africa/Algiers,1715849100,2024-05-16 09:45,...,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.5000,1.5200,Europe/Andorra,1715849100,2024-05-16 10:45,...,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.8400,13.2300,Africa/Luanda,1715849100,2024-05-16 09:45,...,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43879,Venezuela,Caracas,10.5000,-66.9167,America/Caracas,1735383600,2024-12-28 07:00,...,2,06:44 AM,06:15 PM,04:35 AM,04:16 PM,Waning Crescent,8
43880,Vietnam,Hanoi,21.0333,105.8500,Asia/Bangkok,1735383600,2024-12-28 18:00,...,10,06:32 AM,05:25 PM,04:00 AM,02:58 PM,Waning Crescent,11
43881,Yemen,Sanaa,15.3547,44.2067,Asia/Aden,1735380900,2024-12-28 13:15,...,2,06:28 AM,05:42 PM,04:04 AM,03:26 PM,Waning Crescent,10
43882,Zambia,Lusaka,-15.4167,28.2833,Africa/Lusaka,1735383600,2024-12-28 13:00,...,1,05:38 AM,06:40 PM,03:09 AM,04:37 PM,Waning Crescent,9


---
---

# Question 1: Outlier Detection Methods (35 points)

Implement both IQR and Z-score outlier detection, then compare results on temperature data.

---

In [ ]:
def detect_outliers_iqr(series, multiplier=1.5):
    """
    Detect outliers using IQR method.
    Returns dict with: outlier_mask, outlier_count, Q1, Q3, IQR, lower_fence, upper_fence
    """
    # ============================================
    # YOUR CODE HERE
    # ============================================
    
    Q1 = None  # series.quantile(0.25)
    Q3 = None  # series.quantile(0.75)
    IQR = None  # Q3 - Q1
    lower_fence = None  # Q1 - multiplier * IQR
    upper_fence = None  # Q3 + multiplier * IQR
    outlier_mask = None  # (series < lower_fence) | (series > upper_fence)
    
    # ============================================
    
    return {
        'outlier_mask': outlier_mask,
        'outlier_count': outlier_mask.sum(),
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'lower_fence': lower_fence, 'upper_fence': upper_fence
    }


def detect_outliers_zscore(series, threshold=3.0):
    """
    Detect outliers using Z-score method.
    Returns dict with: outlier_mask, outlier_count, mean, std, z_scores
    """
    # ============================================
    # YOUR CODE HERE
    # ============================================
    
    mean = None  # series.mean()
    std = None  # series.std()
    z_scores = None  # (series - mean) / std
    outlier_mask = None  # np.abs(z_scores) > threshold
    
    # ============================================
    
    return {
        'outlier_mask': outlier_mask,
        'outlier_count': outlier_mask.sum(),
        'mean': mean, 'std': std, 'z_scores': z_scores
    }

In [ ]:
# Test both methods on temperature
iqr_result = detect_outliers_iqr(df_weather['temperature_celsius'])
zscore_result = detect_outliers_zscore(df_weather['temperature_celsius'])

print("IQR Method:")
print(f"  Outliers: {iqr_result['outlier_count']:,}")
print(f"  Fences: [{iqr_result['lower_fence']:.1f}, {iqr_result['upper_fence']:.1f}]")

print("\nZ-Score Method:")
print(f"  Outliers: {zscore_result['outlier_count']:,}")
print(f"  Mean: {zscore_result['mean']:.1f}, Std: {zscore_result['std']:.1f}")

---
---

# Question 2: Domain Validator Class (35 points)

Create a class that validates weather data against physical constraints.

---

In [ ]:
class DomainValidator:
    """
    Validates data against domain-specific rules.
    """
    
    RULES = {
        'temperature_celsius': {'min': -90, 'max': 60},
        'humidity': {'min': 0, 'max': 100},
        'wind_kph': {'min': 0, 'max': 410},
        'pressure_mb': {'min': 870, 'max': 1085},
    }
    
    def __init__(self, custom_rules=None):
        self.rules = self.RULES.copy()
        if custom_rules:
            self.rules.update(custom_rules)
    
    def validate_column(self, series, column_name):
        """
        Validate column against rules.
        Returns dict with: violation_count, violation_pct, actual_min, actual_max
        """
        # ============================================
        # YOUR CODE HERE
        # ============================================
        
        if column_name not in self.rules:
            return {'status': 'no_rule'}
        
        rule = self.rules[column_name]
        below_min = None  # series < rule['min']
        above_max = None  # series > rule['max']
        violations = None  # below_min | above_max
        
        # ============================================
        
        return {
            'column': column_name,
            'valid_range': f"[{rule['min']}, {rule['max']}]",
            'violation_count': violations.sum() if violations is not None else 0,
            'actual_min': series.min(),
            'actual_max': series.max()
        }
    
    def validate_dataframe(self, df):
        """
        Validate all columns that have rules.
        """
        # ============================================
        # YOUR CODE HERE: Loop through columns and validate
        # ============================================
        
        results = []
        # YOUR CODE HERE
        
        # ============================================
        return results

In [ ]:
# Test DomainValidator
validator = DomainValidator()
results = validator.validate_dataframe(df_weather)

print("Domain Validation Results")
print("=" * 50)
for r in results:
    print(f"{r['column']}: {r['violation_count']} violations")

---
---

# Question 3: Data Quality Scorer (30 points)

Build a quality scorer that calculates completeness, validity, and uniqueness scores.

---

In [ ]:
class DataQualityScorer:
    """
    Calculates data quality scores.
    """
    
    def __init__(self, df):
        self.df = df
        self.scores = {}
    
    def score_completeness(self):
        """
        Score = (1 - missing_cells / total_cells) * 100
        """
        # ============================================
        # YOUR CODE HERE
        # ============================================
        
        total = len(self.df) * len(self.df.columns)
        missing = None  # self.df.isnull().sum().sum()
        self.scores['completeness'] = None  # (1 - missing/total) * 100
        
        # ============================================
        return self
    
    def score_uniqueness(self):
        """
        Score = (1 - duplicates / total_rows) * 100
        """
        # ============================================
        # YOUR CODE HERE
        # ============================================
        
        duplicates = None  # self.df.duplicated().sum()
        self.scores['uniqueness'] = None  # (1 - duplicates/len(self.df)) * 100
        
        # ============================================
        return self
    
    def calculate_overall(self):
        """
        Overall = average of all scores
        """
        if self.scores:
            self.scores['overall'] = sum(self.scores.values()) / len(self.scores)
        return self
    
    def get_report(self):
        return self.scores

In [ ]:
# Test DataQualityScorer
scores = (
    DataQualityScorer(df_weather)
    .score_completeness()
    .score_uniqueness()
    .calculate_overall()
    .get_report()
)

print("Data Quality Scores")
print("=" * 30)
for dim, score in scores.items():
    if score:
        print(f"{dim.capitalize():15} {score:.1f}%")

---

## Verification

In [ ]:
print("Lab 5 Verification")
print("=" * 40)

# Q1
try:
    r = detect_outliers_iqr(df_weather['temperature_celsius'])
    assert r['Q1'] is not None
    print("✓ Q1: IQR detection works")
except: print("✗ Q1: IQR detection failed")

try:
    r = detect_outliers_zscore(df_weather['temperature_celsius'])
    assert r['mean'] is not None
    print("✓ Q1: Z-score detection works")
except: print("✗ Q1: Z-score detection failed")

# Q2
try:
    v = DomainValidator()
    r = v.validate_dataframe(df_weather)
    assert len(r) > 0
    print("✓ Q2: DomainValidator works")
except: print("✗ Q2: DomainValidator failed")

# Q3
try:
    s = DataQualityScorer(df_weather).score_completeness().get_report()
    assert 'completeness' in s
    print("✓ Q3: DataQualityScorer works")
except: print("✗ Q3: DataQualityScorer failed")

---

## Submission Checklist

- [ ] Q1: Both outlier detection functions work
- [ ] Q2: DomainValidator validates columns correctly
- [ ] Q3: DataQualityScorer calculates scores

Submit via Canvas by **Thursday, February 05, 2026**

---